# Constitutional AI: 헌법적 AI - 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습

- Tutorial ID: `expand-constitutional-ai`
- Tutorial: Constitutional AI: 헌법적 AI
- Section ID: `expand-constitutional-ai-code-2`
- Section: 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습

## 이 노트북에서 배우는 것

Constitutional AI 논문(Bai et al., 2022)의 학습 과정은 크게 두 단계로 이루어져 있습니다.

1. **1단계 · SL-CAI (지도학습)** — 모델이 스스로 자기 답변을 비판(critique)하고 다시 쓰는(revise) 단계
2. **2단계 · RLAIF (이번 노트북)** — **사람이 아니라 AI가** 미리 정해둔 원칙(헌법)에 따라 "두 응답 중 어느 쪽이 더 나은가"를 판단하고, 그 판단 결과(선호도 데이터)로 보상 모델(reward model)을 학습시키는 단계

이번 노트북은 2단계, 즉 **"사람 라벨러 대신 AI가 응답 쌍을 비교 평가한다"**는 RLAIF의 핵심 아이디어를 코드로 처음부터 끝까지 따라가 보는 실습입니다.

> **선수 지식**: Python 기초 문법과 "언어 모델이 다음 단어를 확률적으로 예측한다"는 정도의 개념이면 충분합니다. temperature, 선호도(preference) 데이터, 보상 모델 등 나머지 개념은 노트북 안에서 하나씩 설명합니다.
>
> **준비물**: OpenAI API 키 (`gpt-4` 모델 호출에 사용하며, 유료입니다)

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# "사람 라벨러 대신 AI가 두 응답을 비교 평가한다"는 RLAIF 개념이
# 실제 함수 호출과 데이터 구조로 어떻게 구현되는지 한 단계씩 추적하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature의 효과를 직접 계산해서 관찰한다
#   2) "헌법(원칙)"이 AI 심판의 판단 기준으로 어떻게 사용되는지 이해한다
#   3) (prompt, chosen, rejected) 형태의 선호도 데이터가 어떻게 만들어지는지 이해한다
#   4) 이 선호도 데이터가 보상 모델(reward model) 학습에 어떻게 쓰이는지 개념적으로 이해한다
#
# 읽는 순서:
#   1) 온도(temperature) 미니 실험 셀을 먼저 실행해서 "temperature ↔ 확률분포" 감각을 잡습니다.
#   2) PRINCIPLES(헌법) 정의를 읽고, 이 원칙들이 어디서(judge 프롬프트) 쓰이는지 확인합니다.
#   3) generate_two_responses → judge_responses → parse_judgment 순서로
#      함수 하나하나가 어떤 입력을 받아 어떤 출력을 내는지 확인합니다.
#   4) 위 3개 함수를 묶은 generate_preference_pairs가 반복문 안에서
#      어떻게 (prompt, chosen, rejected) 딕셔너리를 쌓아가는지 확인합니다.
#   5) n_pairs, temperature 값, PRINCIPLES 내용을 바꿔가며 결과가 어떻게 달라지는지 실험해봅니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "정보가 어떤 형태로 바뀌며 다음 단계로 넘어가는지"를 보세요.
#     (문자열 프롬프트 → 응답 2개 → 심판의 판단 텍스트 → A/B 라벨 → chosen/rejected 딕셔너리)
#   - openai 패키지를 사용하는 셀은 유료 API 호출이 발생합니다. 실행 전 OPENAI_API_KEY를 설정하세요.
#   - "[심화]"와 "[보너스]" 표시가 붙은 셀은 핵심 개념(RLAIF)을 넘어서는 추가 내용입니다. 건너뛰어도 무방합니다.

## 1. RLHF와 RLAIF, 무엇이 다를까?

기존의 **RLHF (Reinforcement Learning from Human Feedback)**는 다음과 같은 순서로 모델을 학습시킵니다.

```
프롬프트 → 모델이 응답 2개 생성 → 사람이 "어느 게 더 나은가" 라벨링
   → 그 라벨(선호도 데이터)로 보상 모델(reward model) 학습
   → 보상 모델의 점수를 이용해 정책 모델을 강화학습(PPO 등)으로 미세조정
```

이 방식의 문제는 **"사람이 일일이 라벨링해야 한다"**는 점입니다. 좋은 선호도 데이터를 몇만~몇십만 개 모으려면 많은 인력과 시간, 비용이 필요합니다. 또한 사람마다 판단 기준이 조금씩 달라서 라벨이 일관되지 않을 수도 있습니다.

**RLAIF (Reinforcement Learning from AI Feedback)**는 이 "사람 라벨러" 자리를 **또 다른 AI 모델**로 대체합니다.

```
프롬프트 → 모델이 응답 2개 생성 → AI 심판이 "헌법(원칙)"을 기준으로 "어느 게 더 나은가" 판단
   → 그 판단(선호도 데이터)으로 보상 모델 학습
   → 보상 모델의 점수를 이용해 정책 모델을 강화학습으로 미세조정
```

바뀐 부분은 딱 하나, **"사람이 라벨링" → "AI가 헌법에 따라 라벨링"**입니다. 나머지 파이프라인(선호도 데이터 → 보상 모델 → 강화학습)은 RLHF와 동일합니다. Constitutional AI 논문은 이 방식을 이용해 사람의 라벨링 없이도 안전하고 유용한 모델을 학습시킬 수 있음을 보여주었습니다.

> 이번 노트북은 위 그림의 **"AI 심판이 판단하는 부분"**, 즉 선호도 데이터를 만드는 과정을 직접 구현합니다. 실제 보상 모델 학습(신경망 학습 루프)까지는 다루지 않지만, 노트북 뒷부분의 [심화] 섹션에서 그 원리를 간단한 예시로 확인해봅니다.

In [ ]:
# ── 준비: 라이브러리 불러오기 ──
#
# 이 노트북은 OpenAI API를 사용해 실제로 텍스트를 생성하고 판단합니다.
# 실행 전 아래 방법 중 하나로 API 키를 설정해주세요.
#
#   방법 1) 터미널에서 환경변수로 설정
#           export OPENAI_API_KEY="sk-..."
#
#   방법 2) 노트북 안에서 직접 설정 (다른 사람과 공유하는 노트북에는 키를 남기지 마세요!)
#           import os
#           os.environ["OPENAI_API_KEY"] = "sk-..."
#
# API 키는 https://platform.openai.com/api-keys 에서 발급받을 수 있습니다.
# 주의: gpt-4 API 호출은 유료입니다. 이 노트북을 끝까지 실행하면
#       프롬프트 3개 × 쌍 3개 × 호출 3회(응답 A, 응답 B, 심판) = 총 27회의 API 호출이 발생합니다.
# 모델명("gpt-4")은 예시이니, 계정에서 사용 가능한 모델명으로 자유롭게 바꿔도 됩니다.

from __future__ import annotations  # 최신 타입 힌트 문법(tuple[str, str] 등)을 오래된 Python에서도 쓸 수 있게 해줍니다

from openai import OpenAI
import json
import numpy as np  # temperature가 확률분포를 바꾸는 과정을 직접 계산해보기 위해 사용합니다

client = OpenAI()  # 환경변수 OPENAI_API_KEY를 자동으로 읽어옵니다

## 2. Temperature는 왜, 어떻게 응답을 다르게 만들까?

언어 모델은 다음에 올 단어(정확히는 토큰)를 하나 고를 때, 가능한 모든 후보 단어에 대해 **logit**이라는 원점수(raw score)를 계산합니다. 이 logit은 아직 확률이 아니라서 음수일 수도, 아주 큰 값일 수도 있습니다.

이 logit을 "합이 1인 확률분포"로 바꿔주는 함수가 **softmax**입니다.

$$P(\text{token}_i) = \frac{e^{\text{logit}_i}}{\sum_j e^{\text{logit}_j}}$$

**temperature ($T$)**는 softmax에 들어가기 전 logit을 얼마나 "누그러뜨릴지" 조절하는 값입니다.

$$P(\text{token}_i) = \frac{e^{\text{logit}_i / T}}{\sum_j e^{\text{logit}_j / T}}$$

- **$T$가 작을수록 (0에 가까울수록)**: logit 간의 차이가 확대되어, 가장 점수가 높은 후보 하나에 확률이 몰립니다. → 항상 비슷한, 예측 가능하고 보수적인 답변
- **$T$가 클수록 (1 이상)**: logit 간의 차이가 완만해져서, 점수가 낮았던 후보에도 뽑힐 기회가 생깁니다. → 더 다양하고 창의적이지만 가끔 엉뚱한 답변

바로 다음 셀에서 숫자 5개짜리 장난감 예시로 이 효과를 직접 눈으로 확인해봅니다. 이후 코드에서 응답 A는 `temperature=0.8`(다양하게), 응답 B는 `temperature=0.2`(보수적으로) 생성하는데, 이렇게 일부러 스타일이 다른 두 응답을 만들어야 뒤에서 AI 심판이 "어느 쪽이 더 나은가"를 의미 있게 비교할 수 있습니다.

In [ ]:
# ── temperature가 확률분포를 어떻게 바꾸는지 직접 계산해보기 ──
#
# 실제 LLM은 수만~수십만 개의 토큰 후보를 가지지만,
# 원리를 눈으로 확인하기 위해 "가상의 단어 5개"짜리 장난감 예시로 단순화합니다.
# 아래 logits는 모델이 "다음 단어"로 각 후보에 매긴 원점수(raw score)라고 가정한 값입니다.

words = ["안녕하세요", "반갑습니다", "그래요", "음...", "네"]
logits = np.array([4.0, 3.0, 1.0, 0.5, 2.0])

def softmax_with_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """
    logit 배열을 temperature로 나눈 뒤 softmax를 적용해 확률분포로 바꿉니다.

    temperature가 작을수록(0에 가까울수록) 1등 후보로 확률이 몰리고(뾰족한 분포),
    temperature가 클수록 여러 후보에 확률이 고르게 퍼집니다(완만한 분포).
    """
    scaled_logits = logits / temperature
    # 아래에서 최댓값을 빼주는 것은 결과값(확률)을 바꾸지 않으면서
    # exp() 계산 중 숫자가 너무 커져 overflow가 나는 것을 막아주는 표준적인 안전장치입니다.
    exp_logits = np.exp(scaled_logits - np.max(scaled_logits))
    return exp_logits / np.sum(exp_logits)

for temp in [0.2, 0.8, 1.5]:
    probs = softmax_with_temperature(logits, temp)
    print(f"\n[temperature = {temp}]")
    for w, p in zip(words, probs):
        bar = "■" * int(p * 50)  # 확률 크기를 막대그래프처럼 시각화
        print(f"  {w:8s} {p:.3f}  {bar}")


[temperature = 0.2]
  안녕하세요    0.993  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  반갑습니다    0.007  
  그래요      0.000  
  음...     0.000  
  네        0.000  

[temperature = 0.8]
  안녕하세요    0.712  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  반갑습니다    0.204  ■■■■■■■■■■
  그래요      0.017  
  음...     0.009  
  네        0.058  ■■

[temperature = 1.5]
  안녕하세요    0.498  ■■■■■■■■■■■■■■■■■■■■■■■■
  반갑습니다    0.256  ■■■■■■■■■■■■
  그래요      0.067  ■■■
  음...     0.048  ■■
  네        0.131  ■■■■■■

## 3. "헌법(Constitution)"이란 무엇인가?

Constitutional AI에서 말하는 **헌법(constitution)**은 복잡한 법률 문서가 아니라, **"좋은 응답이란 무엇인가"를 정의하는 자연어 원칙들의 목록**입니다. 예를 들면 이런 것들입니다.

- "더 도움이 되는 답변을 고르세요"
- "사실과 다른 내용을 담지 않은 답변을 고르세요"
- "특정 집단에 대한 편견을 담지 않은 답변을 고르세요"

사람 라벨러에게 "이 두 답변 중 뭐가 나아요?"라고 물으면 사람마다 기준이 조금씩 다를 수 있습니다. 하지만 AI 심판에게 **"이 원칙들을 기준으로 판단하세요"**라고 명시적으로 알려주면, 훨씬 일관된 기준으로 대량의 판단을 빠르게 만들어낼 수 있습니다. 이것이 바로 아래 `PRINCIPLES` 리스트가 하는 역할입니다.

이 원칙들은 잠시 후 `judge_responses` 함수에서 심판 프롬프트에 통째로 삽입되어, AI 심판이 "무엇을 기준으로 A와 B를 비교해야 하는지" 알려주는 역할을 합니다.

In [ ]:
# ── 헌법(Constitution): AI 심판이 따를 원칙들 ──
#
# 아래 각 원칙은 "두 응답 중 어느 것이 더 나은가"를 판단할 때
# AI 심판이 구체적으로 무엇을 기준 삼아야 하는지 알려줍니다.
# 원칙을 추가하거나 문구를 바꿔보면서 심판의 판단이 어떻게 달라지는지
# 직접 실험해보는 것도 좋은 공부가 됩니다.

PRINCIPLES = [
    # 1) 유용성과 무해성: Constitutional AI에서 가장 기본이 되는 축입니다.
    #    예) "정렬 방법을 모르겠어요"라고만 답하기보다,
    #        실제로 동작하는 코드 예시까지 주는 답변이 더 도움이 됩니다.
    "Choose the response that is more helpful and harmless.",

    # 2) 정직성과 투명성: 모르는 것을 아는 척하지 않고,
    #    확실하지 않은 부분은 확실하지 않다고 밝히는 답변을 우대합니다.
    "Choose the response that is more honest and transparent.",

    # 3) 공정성: 성별·인종·연령·직업 등에 대한 편견 섞인 전제를 깔지 않는 답변을 우대합니다.
    "Choose the response that is less biased and more fair.",

    # 4) 안전성: 위험한 행동을 조장하거나 안전하지 않은 정보를 담지 않는 답변을 우대합니다.
    "Choose the response that avoids providing dangerous or unsafe information.",

    # 5) 구체성: "그건 상황에 따라 달라요"처럼 두루뭉술한 답변보다,
    #    구체적인 예시나 실행 가능한 단계를 제시하는 답변을 우대합니다.
    "Choose the response that is more specific and actionable, rather than vague.",
]

## 4. 첫 번째 함수: 비교할 응답 두 개 만들기

RLAIF 파이프라인의 첫 단계는 **같은 프롬프트에 대해 서로 다른 응답 두 개를 만드는 것**입니다. AI 심판이 "어느 게 더 나은가"를 판단하려면, 애초에 비교할 대상이 서로 달라야 의미가 있습니다.

앞서 2절에서 살펴본 것처럼, 이 코드는 서로 다른 두 개의 `temperature`를 사용해서 응답의 "스타일"을 의도적으로 다르게 만듭니다.

| 변수 | temperature | 경향 |
|---|---|---|
| `resp_a` | 0.8 | 더 다양하고 자유로운 표현 |
| `resp_b` | 0.2 | 더 일관되고 보수적인 표현 |

In [ ]:
def generate_two_responses(prompt: str) -> tuple[str, str]:
    """
    같은 프롬프트에 대해 서로 다른 temperature로 두 개의 응답을 생성합니다.

    - resp_a: temperature=0.8 → 더 다양하고 자유로운 답변
    - resp_b: temperature=0.2 → 더 일관되고 보수적인 답변

    두 응답의 "스타일"을 의도적으로 다르게 만들어서,
    바로 다음에 등장하는 AI 심판이 유의미하게 비교할 수 있도록 합니다.
    (만약 두 응답이 거의 똑같다면 "어느 쪽이 더 나은가"를 판단하는 게 의미 없겠죠?)
    """
    resp_a = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
    ).choices[0].message.content

    resp_b = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    ).choices[0].message.content

    return resp_a, resp_b

## 5. 두 번째 함수: AI 심판에게 물어보기

이제 응답 A, B를 실제로 비교 평가할 **심판 프롬프트**를 만듭니다. 이 프롬프트가 바로 RLAIF의 핵심입니다 — 앞서 정의한 `PRINCIPLES`(헌법)를 프롬프트 안에 그대로 넣어서, "이 기준으로 판단하라"고 AI에게 명시적으로 지시합니다.

심판 프롬프트는 크게 세 부분으로 구성됩니다.

1. **기준 제시**: `PRINCIPLES` 목록을 나열해서 "무엇을 기준으로 비교할지" 알려줌
2. **비교 대상 제시**: 원래 프롬프트, 응답 A, 응답 B를 순서대로 보여줌
3. **출력 형식 지정**: "A 또는 B로만 답하라"고 명확히 요청 → 이렇게 해야 이후 코드에서 결과를 파싱하기 쉬워짐

그리고 심판을 호출할 때는 `temperature=0`을 사용합니다. 이는 2절에서 배운 내용과 정확히 반대 방향의 선택입니다 — 판단 기준은 매번 랜덤하게 흔들리면 안 되고, 최대한 일관되고 결정적(deterministic)이어야 하기 때문입니다.

In [ ]:
def build_judge_prompt(prompt: str, resp_a: str, resp_b: str) -> str:
    """
    AI 심판에게 전달할 프롬프트를 만듭니다.

    PRINCIPLES(헌법)를 텍스트로 나열해 프롬프트 맨 앞에 넣는 것이 핵심입니다.
    이렇게 하면 심판 모델이 "무엇을 기준으로 비교해야 하는지" 명확히 알 수 있습니다.
    """
    # 참고: f-string(f"...")의 중괄호 { } 안에는 Python 3.12 이전 버전에서
    # 줄바꿈 문자 "\n"을 직접 쓸 수 없어서, 원래 코드처럼 chr(10)으로 우회하는 경우가 많았습니다.
    # 여기서는 principles_text를 f-string 밖에서 미리 만들어 그런 문제를 아예 피했습니다.
    principles_text = "\n".join(f"- {p}" for p in PRINCIPLES)

    return f"""You are an AI judge following these principles:
{principles_text}

Which response is better for the following prompt?

PROMPT: {prompt}

RESPONSE A: {resp_a}

RESPONSE B: {resp_b}

Answer ONLY "A" or "B" first, then briefly explain why in 1-2 sentences."""


def judge_responses(prompt: str, resp_a: str, resp_b: str) -> str:
    """
    AI 심판 모델을 호출해서 어느 응답이 더 나은지 판단을 받아옵니다.

    temperature=0으로 호출하는 이유:
    판단 기준이 호출할 때마다 달라지면 신뢰할 수 있는 선호도 데이터를 만들 수 없기 때문에,
    가장 결정적인(deterministic) 설정을 사용합니다.
    """
    judge_prompt = build_judge_prompt(prompt, resp_a, resp_b)

    judgment = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0,
    ).choices[0].message.content

    return judgment

## 6. 세 번째 함수: 심판의 답변에서 "A/B"만 뽑아내기

`judge_responses`가 돌려주는 것은 아직 자유 형식의 문장입니다. 예를 들면 다음과 같은 텍스트가 돌아옵니다.

> "A - Response A gives a concrete code example, which is more actionable."

이 문장에서 우리에게 진짜 필요한 정보는 맨 앞의 **"A"** 한 글자뿐입니다. 아래 `parse_judgment` 함수는 문자열이 `"A"`로 시작하는지, `"B"`로 시작하는지만 확인하는 아주 단순한 파싱을 합니다.

> **beginner 참고**: 이렇게 "모델이 시키는 대로 답할 것"이라고 가정하고 문자열 앞부분만 확인하는 방식은 간단하지만 취약합니다. 만약 모델이 "저는 A를 선택하겠습니다"처럼 다른 형식으로 답하면 파싱이 어긋날 수 있습니다. 실무에서는 정규식(regex)으로 더 유연하게 찾거나, 아예 `response_format={"type": "json_object"}` 같은 옵션으로 모델이 `{"winner": "A"}` 형태의 JSON만 반환하도록 강제하는 방법을 많이 씁니다.

In [ ]:
def parse_judgment(judgment: str) -> str:
    """
    심판 모델의 자유 형식 응답에서 "A" 또는 "B" 판정만 뽑아냅니다.

    build_judge_prompt에서 '"A" 또는 "B"를 먼저 답하라'고 명시했기 때문에,
    보통은 응답의 맨 앞 글자만 확인해도 충분합니다.
    """
    cleaned = judgment.strip()

    if cleaned.startswith("A"):
        return "A"
    elif cleaned.startswith("B"):
        return "B"
    else:
        # 모델이 형식을 지키지 않은 경우를 위한 안전장치입니다.
        # 실전 데이터 파이프라인에서는 이런 샘플은 버리거나(discard) 다시 요청하는 것이 더 안전합니다.
        return "A" if "A" in cleaned[:5] else "B"

## 7. 네 단계를 하나로 합치기: `generate_preference_pairs`

지금까지 만든 세 함수를 순서대로 실행하면 "선호도 쌍(preference pair)" 하나가 완성됩니다.

```
generate_two_responses  →  resp_a, resp_b
judge_responses          →  judgment (자유 형식 텍스트)
parse_judgment            →  "A" 또는 "B"
```

마지막으로 판정 결과에 따라 `chosen`(선택된 응답)과 `rejected`(기각된 응답)이라는 이름을 붙여 딕셔너리로 저장합니다. 이 과정을 `n_pairs`번 반복해서 하나의 프롬프트에 대해 여러 개의 선호도 쌍을 만듭니다.

In [ ]:
def generate_preference_pairs(prompt: str, n_pairs: int = 5) -> list[dict]:
    """
    RLAIF: AI가 헌법에 따라 응답 쌍의 선호도를 판단합니다.

    하나의 prompt에 대해 n_pairs번 반복하면서,
    매번 (응답 생성 → 심판 → 파싱) 과정을 거쳐 선호도 쌍을 하나씩 쌓습니다.

    반환값 형식: [{"prompt": ..., "chosen": ..., "rejected": ..., "judgment": ...}, ...]
    이 형식은 뒤에서 DPO 학습 데이터로 그대로 사용됩니다.
    """
    pairs = []

    for i in range(n_pairs):
        # 1) 서로 다른 temperature로 응답 2개 생성
        resp_a, resp_b = generate_two_responses(prompt)

        # 2) AI 심판에게 어느 쪽이 더 나은지 판단받기 (RLAIF 핵심)
        judgment = judge_responses(prompt, resp_a, resp_b)

        # 3) 판단 텍스트에서 "A"/"B" 라벨만 추출
        winner = parse_judgment(judgment)

        # 4) 판정에 따라 chosen(선택됨) / rejected(기각됨) 라벨 부여
        chosen = resp_a if winner == "A" else resp_b
        rejected = resp_b if winner == "A" else resp_a

        pairs.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "judgment": judgment,
        })

        print(f"  [{i + 1}/{n_pairs}] 판정 결과: {winner}")

    return pairs

## 8. 실제로 실행하기

이제 서로 다른 성격의 프롬프트 3개(코딩 / 건강 조언 / 사실 정보)에 대해, 프롬프트당 3개씩 선호도 쌍을 생성합니다.

**예상 API 호출 횟수**: 프롬프트 3개 × 쌍 3개 × (응답 A + 응답 B + 심판, 호출 3회) = **총 27회**

`gpt-4` 기준 호출 1회에 수 초가 걸릴 수 있어, 전체 실행에는 1~2분 정도 소요될 수 있습니다. 아래 셀을 실행하면 각 쌍이 만들어질 때마다 판정 결과가 순서대로 출력됩니다.

In [ ]:
# ── 선호도 데이터 생성 대상 프롬프트 ──
# 서로 다른 성격의 질문(코딩 / 건강 조언 / 사실 정보)을 섞어서
# 다양한 상황에서 헌법 기반 판단이 어떻게 작동하는지 확인합니다.
prompts = [
    "파이썬으로 리스트 정렬하는 방법 알려줘",   # 코딩 질문 → "구체성" 원칙이 두드러질 가능성
    "건강한 식단에 대해 조언해 줘",              # 조언 질문 → "안전성/정직성" 원칙이 두드러질 가능성
    "기후변화의 주요 원인은 뭐야?",              # 사실 정보 질문 → "정직성/공정성" 원칙이 두드러질 가능성
]

all_pairs = []
for p in prompts:
    print(f"\n=== 프롬프트: {p} ===")
    all_pairs.extend(generate_preference_pairs(p, n_pairs=3))

print(f"\n총 {len(all_pairs)}개의 선호도 쌍(preference pair)이 생성되었습니다.")

## 9. DPO 학습용 데이터셋으로 저장하기

`all_pairs`에서 `prompt`, `chosen`, `rejected` 세 필드만 남긴 것이 바로 **DPO(Direct Preference Optimization)** 학습에 그대로 사용할 수 있는 표준 데이터 형식입니다. (`judgment` 필드는 사람이 데이터를 검수할 때 참고하도록 남겨둔 것으로, 실제 학습에는 쓰이지 않습니다.)

```json
{"prompt": "...", "chosen": "AI 심판이 더 낫다고 고른 응답", "rejected": "AI 심판이 덜 낫다고 고른 응답"}
```

이 `(prompt, chosen, rejected)` 3종 세트 형식은 Hugging Face `trl` 라이브러리의 `DPOTrainer`를 비롯한 대부분의 선호도 기반 학습 도구가 공통으로 사용하는 형식입니다. DPO가 정확히 무엇을 하는지는 11절([심화])에서 조금 더 다룹니다.

In [ ]:
dpo_dataset = [{
    "prompt": p["prompt"],
    "chosen": p["chosen"],
    "rejected": p["rejected"],
} for p in all_pairs]

# encoding="utf-8"을 명시해야 한글이 깨지지 않고 파일에 안전하게 저장됩니다.
with open("rlaif_preference_data.json", "w", encoding="utf-8") as f:
    json.dump(dpo_dataset, f, indent=2, ensure_ascii=False)

print(f"RLAIF 선호도 데이터 {len(dpo_dataset)}쌍 생성 완료")
print(f"→ 이 데이터로 DPO 또는 보상 모델(RM) 학습이 가능합니다!")
print(f"→ 저장 위치: rlaif_preference_data.json")

In [ ]:
# 생성된 데이터 중 앞 2개만 미리보기
for pair in dpo_dataset[:2]:
    print(f"\n프롬프트: {pair['prompt']}")
    print(f"  ✅ 선택됨(chosen):  {pair['chosen'][:80]}...")
    print(f"  ❌ 기각됨(rejected): {pair['rejected'][:80]}...")

## 10. [심화] 보상 모델은 이 데이터로 정확히 어떻게 학습될까?

지금까지 만든 것은 **데이터**입니다. 실제로 "보상 모델(reward model)"이라고 부르려면, 이 데이터로 신경망을 학습시켜야 합니다. 원리는 다음과 같습니다.

1. 보상 모델 $r_\theta$는 (prompt, response)를 입력받아 점수(scalar) 하나를 출력하는 신경망입니다. (보통 사전학습된 언어모델에 점수 출력용 head를 하나 붙인 구조입니다.)
2. 하나의 선호도 쌍 $(x, y_w, y_l)$ — $x$: 프롬프트, $y_w$: chosen(승자), $y_l$: rejected(패자) — 에 대해 다음 손실 함수를 최소화하도록 학습합니다.

$$\mathcal{L} = -\log \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)$$

여기서 $\sigma$는 sigmoid 함수입니다. 이 손실은 **"chosen 점수 − rejected 점수"가 클수록(즉 보상 모델이 심판의 판단에 동의할수록) 작아지고, 반대일 경우 커지는** 형태입니다. 이런 형태의 손실 함수를 **Bradley-Terry 모델**이라고 부르며, 스포츠 랭킹처럼 "둘 중 무엇이 더 나은가"를 비교하는 데이터를 학습할 때 널리 쓰입니다.

바로 아래 코드에서 이 손실 함수를 아주 작은 숫자 예시로 직접 계산해봅니다.

In [ ]:
def bradley_terry_loss(reward_chosen: float, reward_rejected: float) -> float:
    """
    보상 모델 학습에 쓰이는 손실 함수(Bradley-Terry loss)를 계산합니다.

    reward_chosen  : 보상 모델이 "AI 심판이 고른 chosen 응답"에 매긴 점수
    reward_rejected: 보상 모델이 "AI 심판이 버린 rejected 응답"에 매긴 점수

    이 손실을 최소화하도록 학습시키면, 보상 모델은 점점
    "chosen에는 높은 점수, rejected에는 낮은 점수"를 주는 방향으로 조정됩니다.
    """
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    # 아주 작은 값(1e-9)을 더하는 이유: sigmoid(x)가 0에 아주 가까워지면
    # log(0) = -infinity가 되어 계산이 깨지는 것을 막기 위한 안전장치입니다.
    return -np.log(sigmoid(reward_chosen - reward_rejected) + 1e-9)

# 예시 1: 보상 모델이 아직 학습되지 않아 "틀린" 판단을 하는 경우
# (심판은 chosen을 골랐는데, 보상 모델은 반대로 rejected에 더 높은 점수를 준 상황)
loss_before = bradley_terry_loss(reward_chosen=0.1, reward_rejected=0.9)
print(f"[학습 전, 판단이 틀림] chosen=0.1, rejected=0.9 → loss={loss_before:.4f}  (손실이 큼)")

# 예시 2: 보상 모델이 잘 학습되어 심판과 "같은" 판단을 하는 경우
loss_after = bradley_terry_loss(reward_chosen=0.9, reward_rejected=0.1)
print(f"[학습 후, 판단이 맞음] chosen=0.9, rejected=0.1 → loss={loss_after:.4f}  (손실이 작음)")

[학습 전, 판단이 틀림] chosen=0.1, rejected=0.9 → loss=1.1711  (손실이 큼)
[학습 후, 판단이 맞음] chosen=0.9, rejected=0.1 → loss=0.3711  (손실이 작음)

## 11. [심화] 그럼 DPO는 보상 모델과 뭐가 다른가?

전통적인 RLHF/RLAIF 파이프라인은 이렇게 흘러갑니다.

```
선호도 데이터 → ① 보상 모델 학습 → ② PPO 강화학습으로 정책 모델 미세조정
```

이 방식은 두 단계(① 보상 모델, ② 강화학습)를 각각 구현하고 조율해야 해서 복잡하고, 메모리에 정책 모델·보상 모델·기준(reference) 모델 등 여러 모델을 동시에 올려야 하는 경우도 많아 학습이 불안정해지기 쉽습니다.

**DPO(Direct Preference Optimization)**는 "① + ②를 거친 최종 결과"와 수학적으로 **같은 최적해**에 도달하는 손실 함수를 유도해서, 별도의 보상 모델이나 강화학습 루프 없이 **선호도 데이터로 정책 모델을 곧바로 학습**시킵니다.

```
선호도 데이터 → 정책 모델을 곧바로 미세조정 (보상 모델도, PPO도 필요 없음)
```

그래서 앞서 저장한 `(prompt, chosen, rejected)` 형식의 `rlaif_preference_data.json` 파일은, 10절처럼 보상 모델을 직접 학습시키지 않아도 DPO 학습 라이브러리(예: Hugging Face `trl`의 `DPOTrainer`)에 그대로 넣을 수 있습니다. 최근 실무에서 RLAIF/RLHF 데이터를 활용할 때 DPO를 더 자주 쓰는 이유가 여기에 있습니다.

## 12. [보너스] AI 심판도 완벽하지 않다 — 위치 편향(Position Bias)

지금까지 만든 `judge_responses`는 항상 "RESPONSE A"를 먼저, "RESPONSE B"를 나중에 보여주고 판단을 요청합니다. 그런데 LLM-as-judge 관련 연구들에 따르면, AI 심판은 **응답의 실제 품질과 무관하게 특정 위치(예: 먼저 제시된 응답)를 더 선호하는 경향**을 보일 수 있습니다. 이를 **위치 편향(position bias)**이라고 부릅니다.

간단한 개선 방법은 **같은 두 응답을 순서만 바꿔서 두 번 판단**시켜 보는 것입니다.

1. (A=resp_a, B=resp_b) 순서로 판단 → 승자 확인
2. (A=resp_b, B=resp_a) 순서로 판단 → 승자 확인
3. 두 번의 판단이 **같은 실제 응답**을 가리키면 그 결과를 신뢰하고, 서로 다르면 "판단 보류"로 처리

아래 `judge_responses_robust` 함수는 이 아이디어를 구현한 보너스 코드입니다. API 호출이 2배로 늘어나는 대신, 더 신뢰할 수 있는 선호도 데이터를 얻을 수 있습니다. (핵심 흐름을 이해하는 데 꼭 필요하지는 않으니, 시간이 없다면 건너뛰어도 좋습니다.)

In [ ]:
def judge_responses_robust(prompt: str, resp_a: str, resp_b: str) -> tuple[str | None, str]:
    """
    [보너스] 위치 편향(position bias)을 줄이기 위해 순서를 바꿔 두 번 판단하는 심판 함수.

    반환값:
      (승자 응답 텍스트 또는 None, "consistent" 또는 "tie")
      - "consistent": 순서를 바꿔도 같은 응답이 승자로 뽑힌 경우 (신뢰도 높음)
      - "tie"       : 순서를 바꾸니 승자가 뒤바뀐 경우 (위치 편향 의심 → 이 샘플은 보류)
    """
    # 1차 판단: A=resp_a, B=resp_b 순서 그대로
    judgment_1 = judge_responses(prompt, resp_a, resp_b)
    winner_1 = parse_judgment(judgment_1)
    winner_1_response = resp_a if winner_1 == "A" else resp_b

    # 2차 판단: 순서를 뒤바꿔서 A=resp_b, B=resp_a
    judgment_2 = judge_responses(prompt, resp_b, resp_a)
    winner_2 = parse_judgment(judgment_2)
    winner_2_response = resp_b if winner_2 == "A" else resp_a

    # 두 번의 판단이 "같은 실제 응답 내용"을 가리키는지 비교
    if winner_1_response == winner_2_response:
        return winner_1_response, "consistent"
    else:
        return None, "tie"

# 사용 예시 (실행하려면 아래 주석을 해제하세요 - API 호출이 2회 추가로 발생합니다):
# resp_a, resp_b = generate_two_responses(prompts[0])
# winner, status = judge_responses_robust(prompts[0], resp_a, resp_b)
# print(f"결과: {status}, 승자 응답 앞부분: {winner[:50] if winner else 'N/A'}")

## 13. 정리

이번 노트북에서 다룬 내용을 정리하면 다음과 같습니다.

- **RLAIF**는 RLHF의 "사람 라벨러"를 "헌법(원칙)에 따라 판단하는 AI"로 대체한 방법입니다.
- **temperature**는 logit을 확률분포로 바꾸는 softmax 계산에서 분포가 얼마나 뾰족한지를 조절하며, 낮을수록 결정적이고 높을수록 다양한 출력을 만듭니다.
- **헌법(PRINCIPLES)**은 AI 심판이 응답을 비교할 때 사용하는 명시적인 기준입니다.
- **선호도 쌍 (prompt, chosen, rejected)**은 RLAIF의 최종 산출물이며, 보상 모델 학습과 DPO 학습 모두에 사용될 수 있는 표준 형식입니다.
- 보상 모델은 **Bradley-Terry 손실**로 학습되어 chosen에는 높은 점수를, rejected에는 낮은 점수를 주도록 조정됩니다.
- **DPO**는 별도의 보상 모델·강화학습 없이 같은 선호도 데이터로 정책 모델을 직접 최적화하는 최신 대안입니다.
- 실제 파이프라인에서는 **판정 파싱의 견고함**과 **위치 편향** 같은 디테일까지 신경 써야 신뢰할 수 있는 데이터를 얻을 수 있습니다.

### 한계와 다음 실습

이 노트북의 구현은 교육 목적으로 단순화되어 있습니다. 실제로는 다음과 같은 점들을 더 고려해야 합니다.

- 프롬프트/원칙 개수가 매우 적음 (실전에서는 수천~수만 개 프롬프트, 더 정교한 헌법이 필요합니다)
- `parse_judgment`가 형식을 벗어난 응답을 처리하는 방식이 단순함
- 위치 편향 외에도 심판 모델 자체의 편향(judge model bias)을 검증하는 절차가 없음
- 실제 보상 모델 학습(신경망 파라미터 업데이트)이나 DPO 학습 루프 자체는 다루지 않음 (별도 실습에서 `trl`, `transformers` 등을 사용해 다룰 수 있습니다)

다음 실습에서는 이렇게 만든 선호도 데이터를 실제로 `DPOTrainer` 등에 넣어 정책 모델을 미세조정하는 과정을 다뤄볼 수 있습니다.